# Transportation Equity — A Series That Is Not a Partition

The ACS publishes `B08105` nine times, suffixed `A` through `I`, one per racial and ethnic
group. The lettering reads like a series: work through it and you have covered everyone.

Sum all nine and you get 170.6% of the workers they describe.

Nothing in the table names, the variable labels or the API response says this. Each iteration
is internally consistent, each has a sensible universe, and the overflow appears only when
they are added — which is exactly what anyone building a group comparison does first.

## Learning Objectives

1. **Test whether a set of categories partitions its universe** before treating it as one,
   using a check that costs one line.
2. **Distinguish a subset from a sibling** among categories presented identically.
3. **Notice a universe narrower than the obvious denominator**, and identify who it drops.
4. **Report a group comparison without a rate you cannot construct.**

## Prerequisites

- D01 for ACS `E`/`M` conventions; D18 for the unit-mismatch case this one complements.
- Handbook §3 (denominators, universes and partitions).

## Data Sources

| table | concept | universe (Census wording, retrieved at build time) |
|---|---|---|
| `B08301` | Means of Transportation to Work | Workers 16 years and over |
| `B08105A`–`I` | Means of Transportation to Work, by race/ethnicity iteration | e.g. `A`: **White alone** workers 16 years and over |
| `B08141` | Means of Transportation to Work by Vehicles Available | **Workers 16 years and over in households** |

The iteration suffixes: `A` White alone · `B` Black or African American alone · `C` American
Indian and Alaska Native alone · `D` Asian alone · `E` Native Hawaiian and Other Pacific
Islander alone · `F` Some other race alone · `G` Two or more races · `H` White alone, not
Hispanic or Latino · `I` Hispanic or Latino of any race.

In [1]:
"""Configuration."""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

_here = Path.cwd().resolve()
for _cand in (_here, *_here.parents):
    if (_cand / "src" / "krl_tutorials_data").is_dir():
        sys.path.insert(0, str(_cand / "src"))
        break
else:
    raise SystemExit("Could not locate krl_tutorials_data from " + str(_here))

from krl_tutorials_data.sources import fetch_acs

STATE_FIPS = "51"
ACS_YEAR = 2023
#: The repository root, found by walking UP from wherever this runs. Provenance
#: used to be written to `Path.cwd() / "provenance"` in some notebooks and to a
#: repo-root-relative path in others, so the two conventions could not both be
#: satisfied by one working directory -- and a run from the wrong one silently
#: wrote sixteen records into a stray folder at the top of the repo.
REPO_ROOT = next(c for c in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                 if (c / "src" / "krl_tutorials_data").is_dir())
OUT = REPO_ROOT / "projects" / "07-justice-equity/D23_transportation_equity" / "provenance"
ITERATIONS = {
    "A": "White alone", "B": "Black or African American alone",
    "C": "American Indian and Alaska Native alone", "D": "Asian alone",
    "E": "Native Hawaiian and Other Pacific Islander alone", "F": "Some other race alone",
    "G": "Two or more races", "H": "White alone, not Hispanic or Latino",
    "I": "Hispanic or Latino of any race",
}
MUTUALLY_EXCLUSIVE = "ABCDEFG"   # the claim this notebook tests rather than assumes
print(f"Virginia, ACS 5-year {ACS_YEAR - 4}-{ACS_YEAR}, {len(ITERATIONS)} iterations")

Virginia, ACS 5-year 2019-2023, 9 iterations


In [2]:
"""Retrieve the iterations, the parent table, and the vehicles table."""
VARS = {"B08301_001E": "workers 16+", "B08301_001M": "workers 16+ MOE"}
for letter in ITERATIONS:
    VARS[f"B08105{letter}_001E"] = f"workers 16+, {ITERATIONS[letter]}"
    VARS[f"B08105{letter}_001M"] = f"workers 16+, {ITERATIONS[letter]} MOE"
for i in range(1, 6):
    VARS[f"B08141_{i:03d}E"] = f"B08141_{i:03d}"
    VARS[f"B08141_{i:03d}M"] = f"B08141_{i:03d} MOE"

eq, prov = fetch_acs(year=ACS_YEAR, variables=VARS,
                     geography="county:*", within=f"state:{STATE_FIPS}")
eq["fips"] = eq["state"] + eq["county"]
written = [prov.write(OUT / "d23_acs_transportation_equity.json")]
print(f"{len(eq)} counties and independent cities")
print()
print(prov.render()[:420])

133 counties and independent cities

Source      : U.S. Census Bureau - American Community Survey (acs/acs5)
Vintage     : 2023  (pinned)
Endpoint    : https://api.census.gov/data/2023/acs/acs5
Retrieved   : 2026-09-18T12:13:10+00:00
Shape       : 133 rows x 33 cols
Payload hash: 24c6b2311907bd8b...  (canonical-rows)
Data mode   : EMPIRICAL
Variables   :
              B08301_001E      workers 16+
              B08301_001M      workers 16+ MOE
          


## 1. The test that costs one line

A set of categories partitions a universe if the parts sum to the whole. That is checkable,
and it is checkable *before* any analysis depends on the answer.

In [3]:
"""Do the iterations sum to the parent total? Test A-G and A-I separately."""
total = eq.B08301_001E.sum()
part = {L: eq[f"B08105{L}_001E"].sum() for L in ITERATIONS}

sum_ag = sum(part[L] for L in MUTUALLY_EXCLUSIVE)
sum_ai = sum(part.values())

print(f"{'B08301_001  all workers 16+':52} {total:>12,.0f}")
for L, label in ITERATIONS.items():
    marker = "   <- claimed mutually exclusive" if L in MUTUALLY_EXCLUSIVE else ""
    print(f"  B08105{L}_001  {label:38} {part[L]:>12,.0f}{marker}")
print()
print(f"{'sum of A..G':52} {sum_ag:>12,.0f}   {100 * sum_ag / total:7.2f}% of total")
print(f"{'sum of A..I':52} {sum_ai:>12,.0f}   {100 * sum_ai / total:7.2f}% of total")
print()
print(f"A..G reproduces the parent total to within {abs(sum_ag - total):,.0f} workers.")
print(f"Adding H and I overshoots by {sum_ai - total:,.0f} -- about {100 * (sum_ai - total) / total:.0f}% of the workforce.")

B08301_001  all workers 16+                             4,308,387
  B08105A_001  White alone                               2,692,289   <- claimed mutually exclusive
  B08105B_001  Black or African American alone             782,142   <- claimed mutually exclusive
  B08105C_001  American Indian and Alaska Native alone       13,906   <- claimed mutually exclusive
  B08105D_001  Asian alone                                 321,471   <- claimed mutually exclusive
  B08105E_001  Native Hawaiian and Other Pacific Islander alone        3,141   <- claimed mutually exclusive
  B08105F_001  Some other race alone                       177,784   <- claimed mutually exclusive
  B08105G_001  Two or more races                           317,654   <- claimed mutually exclusive
  B08105H_001  White alone, not Hispanic or Latino       2,571,394
  B08105I_001  Hispanic or Latino of any race              470,299

sum of A..G                                             4,308,387    100.00% of total
sum of A.

**A through G is a genuine partition.** Every worker has exactly one entry among "race alone"
and "two or more races", so those seven sum to the parent total — here exactly.

**H and I are not siblings of A–G; they cut across them.** `H` (White alone, not Hispanic) is
a *subset* of `A` (White alone). `I` (Hispanic or Latino) is an ethnicity question answered
independently of race, so its members are already counted somewhere in A–G. The nine
iterations are two overlapping schemes published under one lettering.

The check below makes the subset relationship explicit rather than resting on the label.

In [4]:
"""H must be a subset of A, and I must overlap A-G. Verify per county, not statewide."""
h_exceeds_a = (eq.B08105H_001E > eq.B08105A_001E).sum()
implied_hispanic_white = eq.B08105A_001E - eq.B08105H_001E

print(f"Counties where H (White alone, not Hispanic) exceeds A (White alone): "
      f"{h_exceeds_a} of {len(eq)}")
print("A subset can never exceed its parent, so any county here would be a contradiction")
print("in the data rather than a difference of definition.")
print()
print(f"A - H, statewide: {implied_hispanic_white.sum():,.0f} workers who are White alone")
print("AND Hispanic or Latino. Those workers appear in A and in I, which is why A..I")
print("overshoots -- they are counted twice by construction, not by error.")
print()
neg = (implied_hispanic_white < 0).sum()
print(f"Counties where A - H is negative: {neg} (expected 0)")

Counties where H (White alone, not Hispanic) exceeds A (White alone): 0 of 133
A subset can never exceed its parent, so any county here would be a contradiction
in the data rather than a difference of definition.

A - H, statewide: 120,895 workers who are White alone
AND Hispanic or Latino. Those workers appear in A and in I, which is why A..I
overshoots -- they are counted twice by construction, not by error.

Counties where A - H is negative: 0 (expected 0)


## 2. The denominator that drops the people most likely to be counted

`B08141` reports means of transport by vehicles available — the table any zero-vehicle
analysis needs. Its universe is not "workers 16 and over". It is **"workers 16 years and over
in households"**.

Workers living in group quarters — barracks, dormitories, correctional and residential care
facilities — are outside it. Virginia has a large military and student population, and those
are also populations with unusually low vehicle access. The exclusion is not random with
respect to the quantity being measured.

In [5]:
"""How many workers does the vehicles table not cover, and what does that do to the rate?"""
workers_all = eq.B08301_001E.sum()
workers_hh = eq.B08141_001E.sum()
no_vehicle = eq.B08141_002E.sum()
gap = workers_all - workers_hh

print(f"B08301_001  workers 16+                      {workers_all:>12,.0f}")
print(f"B08141_001  workers 16+ IN HOUSEHOLDS        {workers_hh:>12,.0f}")
print(f"            difference (group quarters)      {gap:>12,.0f}   "
      f"{100 * gap / workers_all:.2f}% of workers")
print()
print(f"B08141_002  workers with NO vehicle available {no_vehicle:>11,.0f}")
print()
print(f"zero-vehicle rate on the CORRECT denominator   {100 * no_vehicle / workers_hh:6.3f}%")
print(f"zero-vehicle rate on B08301 (wrong universe)   {100 * no_vehicle / workers_all:6.3f}%")
print()
print("The two differ by little, because the excluded group is small. That is a fact about")
print("Virginia's numbers, not a licence: the excluded workers are disproportionately")
print("carless, so the correct denominator is the smaller one AND the numerator is missing")
print("its most relevant cases. The error runs the same direction on both sides of the")
print("fraction, and this table cannot quantify it.")

B08301_001  workers 16+                         4,308,387
B08141_001  workers 16+ IN HOUSEHOLDS           4,246,369
            difference (group quarters)            62,018   1.44% of workers

B08141_002  workers with NO vehicle available     121,634

zero-vehicle rate on the CORRECT denominator    2.864%
zero-vehicle rate on B08301 (wrong universe)    2.823%

The two differ by little, because the excluded group is small. That is a fact about
Virginia's numbers, not a licence: the excluded workers are disproportionately
carless, so the correct denominator is the smaller one AND the numerator is missing
its most relevant cases. The error runs the same direction on both sides of the
fraction, and this table cannot quantify it.


In [6]:
"""The vehicles-available distribution, checked for closure."""
labels = {2: "no vehicle", 3: "1 vehicle", 4: "2 vehicles", 5: "3 or more vehicles"}
parts = {labels[i]: eq[f"B08141_{i:03d}E"].sum() for i in labels}
closure = sum(parts.values())

for name, value in parts.items():
    print(f"  {name:20} {value:>12,.0f}   {100 * value / workers_hh:5.2f}%")
print(f"  {'sum':20} {closure:>12,.0f}   {100 * closure / workers_hh:5.2f}% of B08141_001")
print()
print("This partition closes exactly, which is what a partition should do and what §1's")
print("A..I did not. Checking is cheap; the cost is entirely in not checking.")

  no vehicle                121,634    2.86%
  1 vehicle                 825,804   19.45%
  2 vehicles              1,689,254   39.78%
  3 or more vehicles      1,609,677   37.91%
  sum                     4,246,369   100.00% of B08141_001

This partition closes exactly, which is what a partition should do and what §1's
A..I did not. Checking is cheap; the cost is entirely in not checking.


## 3. What can be reported

The iterations do support group comparison — provided each group is compared against **its
own** denominator rather than against a sum of the nine.

What cannot be built from this table alone is a rate whose denominator is "all workers of
group X" where X spans both schemes. Asking "what share of Hispanic workers have no vehicle"
requires `B08141` iterated by ethnicity, which the ACS publishes as a separate table family —
not something to be assembled by subtraction from what is here.

In [7]:
"""Transit share by iteration, each against its own universe."""
rows = []
for L, label in ITERATIONS.items():
    universe = eq[f"B08105{L}_001E"].sum()
    rows.append({
        "iteration": L, "group": label,
        "workers 16+": f"{universe:,.0f}",
        "share of state workforce": f"{100 * universe / total:5.2f}%",
        "in A..G partition": "yes" if L in MUTUALLY_EXCLUSIVE else "no - overlaps",
    })
print(pd.DataFrame(rows).to_string(index=False))
print()
print("Each row's denominator is its own universe. The final column is the part a reader")
print("needs in order to know which rows may be added and which may not.")

iteration                                            group workers 16+ share of state workforce in A..G partition
        A                                      White alone   2,692,289                   62.49%               yes
        B                  Black or African American alone     782,142                   18.15%               yes
        C          American Indian and Alaska Native alone      13,906                    0.32%               yes
        D                                      Asian alone     321,471                    7.46%               yes
        E Native Hawaiian and Other Pacific Islander alone       3,141                    0.07%               yes
        F                            Some other race alone     177,784                    4.13%               yes
        G                                Two or more races     317,654                    7.37%               yes
        H              White alone, not Hispanic or Latino   2,571,394                  

In [8]:
"""FINDINGS"""
print("FINDINGS")
print("=" * 78)
print(f"""
1. B08105's nine iterations are not one series. A..G partition the parent total
   exactly ({sum_ag:,.0f} against {total:,.0f}). Adding H and I gives
   {100 * sum_ai / total:.1f}% of the workforce, because H is a SUBSET of A and I
   cuts across all race categories. Nothing in the table names or variable
   labels signals this; it appears only on summation.

2. {implied_hispanic_white.sum():,.0f} Virginia workers are White alone AND Hispanic or
   Latino, so they appear in both A and I. H never exceeds A in any of the
   {len(eq)} counties, and A - H is never negative, which is what a subset
   relationship requires and is worth verifying rather than assuming.

3. B08141 -- the table any vehicle-access analysis needs -- covers workers IN
   HOUSEHOLDS only, excluding {gap:,.0f} Virginia workers in group quarters
   ({100 * gap / workers_all:.2f}%). Those are disproportionately the carless, so the
   exclusion is not random with respect to the measured quantity, and the
   table cannot quantify its own bias.

4. The zero-vehicle share is {100 * no_vehicle / workers_hh:.3f}% on the correct denominator and
   {100 * no_vehicle / workers_all:.3f}% on the wrong one. The closeness is a fact about
   Virginia's group-quarters share, not a reason to use either denominator
   carelessly elsewhere.

5. The vehicles-available categories DO close exactly against their universe.
   The difference between a partition that closes and a lettering that looks
   like one is a single summation.
""")
print(f"Provenance written: {len(written)} record(s)")
for p in written:
    print(f"  {p}")

FINDINGS

1. B08105's nine iterations are not one series. A..G partition the parent total
   exactly (4,308,387 against 4,308,387). Adding H and I gives
   170.6% of the workforce, because H is a SUBSET of A and I
   cuts across all race categories. Nothing in the table names or variable
   labels signals this; it appears only on summation.

2. 120,895 Virginia workers are White alone AND Hispanic or
   Latino, so they appear in both A and I. H never exceeds A in any of the
   133 counties, and A - H is never negative, which is what a subset
   relationship requires and is worth verifying rather than assuming.

3. B08141 -- the table any vehicle-access analysis needs -- covers workers IN
   HOUSEHOLDS only, excluding 62,018 Virginia workers in group quarters
   (1.44%). Those are disproportionately the carless, so the
   exclusion is not random with respect to the measured quantity, and the
   table cannot quantify its own bias.

4. The zero-vehicle share is 2.864% on the correct denom

## Key Takeaways

1. **Sum the parts before trusting the set.** One line separates a partition from a
   collection, and the ACS iteration lettering gives no other warning.
2. **A subset can be published beside its parent** under an identical naming scheme. `H` sits
   inside `A`; `I` crosses all of A–G.
3. **Read the universe on every table separately**, even within one subject area. `B08301` and
   `B08141` differ by group-quarters residents.
4. **An exclusion that correlates with the outcome is worse than a small exclusion looks.**
   Group-quarters workers are few and are exactly the population a vehicle-access study cares
   about most.

## Exercises

1. **Find the counties where it matters most.** Compute `A - H` as a share of `A` per county
   and identify where the double-count in a naive `A..I` sum would be largest. Does that
   track anything you would have predicted?
2. **Test the partition at a finer level.** §1 checks the `_001` totals. Repeat the closure
   test on a mode — say public transportation — and report whether A–G still sums exactly.
3. **Bound the group-quarters bias.** Assume workers in group quarters have zero-vehicle rates
   of 10%, 25% and 50%. Recompute the statewide zero-vehicle share under each and state how
   much the published figure could understate it.
4. **Carry the margins.** Every figure here is a point estimate. The iteration totals for
   small groups (`C`, `E`) have large relative MOEs — recompute their shares with intervals
   and report which comparisons survive.
5. **Locate the missing table.** Find the ACS table that reports vehicle availability by
   race or ethnicity, and confirm whether it can answer the question §3 says this one cannot.

## References

- U.S. Census Bureau ACS group metadata for `B08301`, `B08105A`–`I` and `B08141`, retrieved
  at build time. The universe wording quoted here comes from that response rather than from
  documentation transcribed by hand.
- Handbook §3 (denominators, universes and partitions); D18 for the complementary case where
  the units rather than the categories mismatch.

## Limitations

- **One state, one vintage.** The partition structure is a property of the ACS and holds
  everywhere; the magnitudes are Virginia's 2019–2023.
- **Point estimates.** MOEs are retrieved and pinned in provenance but not propagated.
  Iterations `C` and `E` are small enough that their shares carry wide intervals, and
  exercise 4 asks for them.
- **The group-quarters bias is argued, not measured.** That those workers have lower vehicle
  access is a reasonable expectation given who lives in barracks and dormitories; this source
  contains no information about them at all, so the direction is asserted and the magnitude
  is unknown. Exercise 3 bounds it under assumptions rather than resolving it.
- **"Vehicles available" is a household attribute, not a personal one.** A worker in a
  three-vehicle household may have access to none of them. The table cannot distinguish
  availability from access.
- **Not tested here:** whether the A–G closure holds at tract level, where small counts and
  disclosure avoidance could break an identity that holds at county level.